In [1]:
print("hello")

hello


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.chat_models.base import init_chat_model
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, List
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from config.env import OPENAI_API_KEY

In [3]:
class StateSchema(TypedDict):
    resume_url: str
    resume_text: str
    Parsed_resume: dict

In [4]:
def pdf_loader(state: StateSchema) -> StateSchema:
    url = state['resume_url']
    loader = PyPDFLoader(url)
    documents = loader.load()
    full_text = "\n".join([doc.page_content for doc in documents])
    state['resume_text'] = full_text
    return state


In [5]:
pdf_loader({ 'resume_url': 'https://res.cloudinary.com/hiregenx/image/upload/v1758444270/s3orzdkufvum3tnydcva.pdf' })

{'resume_url': 'https://res.cloudinary.com/hiregenx/image/upload/v1758444270/s3orzdkufvum3tnydcva.pdf',
 'resume_text': 'MUHAMMAD AHMAD\nFULL STACK DEVELOPER\nLahore, Pakistan ahmaddev477@gmail.com + 92 3412392664 LinkedIn Github\nSUMMARY\nComputer Science undergraduate (6th semester) at COMSATS University Islamabad, Lahore with hands-on experience in\nFullStack development using JavaScript, React.js, Node.js, FastApi and MongoDB. Passionate about building scalable web\napplications and exploring applications of AI/ML (GEN AI). Actively seeking an internship opportunity to apply my skills and\ngrow in a\nprofessional environment.\nSKILLS\nProgramming languages: JavaScript, Python, Rust\nDatabase: MySQL, PostgreSQL, MongoDB.\nFrontend Technologies: React Js, Tailwind Css.\nBackend Technologies: Node.Js, Express Js, FastApi.\nVersion Control: Git.\nSoft Skills: Problem-Solving,Leadership Qualities, Time Management, Teamwork.\nPROJECT EXPERIENCE\nFull Stack Developer (Freelance)\nRemote |

In [26]:
RESUME_PROMPT = """

You are an expert resume parser. Extract the following information from the resume text provided:
- Full Name
- Email Address
- Phone Number
- LinkedIn Profile URL
- GitHub Profile URL
- Portfolio URL
- Professional Summary
- Skills (list)
- Work Experience (list of jobs with company name, position, start date, end date, and description)
- Education (list of institutions with institution name, degree, start year, end year)
- Projects (list of projects with name, description, link, and technologies used)
- Certifications (list of certifications with name, issuer, and year)

and also provide the following AI-generated assessments about the resume quality based on the extracted information:
- aiScore (a float value between 0 and 100 representing the AI's assessment of the resume quality)
- aiSuggestions (list of strings with suggestions for improving the resume)


Ensure that the JSON is properly formatted and valid. If any information is missing in the resume, use null or an empty list as appropriate.
Respond only with the JSON object and no additional text.


{format_instructions} 

"""

In [27]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", RESUME_PROMPT),
    ("human", "Extract the information from this resume text : {resume_text}"),
])


In [28]:

class Experience(BaseModel):
    company: Optional[str] = None
    position: Optional[str] = None
    startDate: Optional[str] = None
    endDate: Optional[str] = None
    description: Optional[str] = None

class Education(BaseModel):
    institution: Optional[str] = None
    degree: Optional[str] = None
    startYear: Optional[str] = None
    endYear: Optional[str] = None

class Project(BaseModel):
    name: Optional[str] = None
    description: Optional[str] = None
    link: Optional[str] = None
    technologies: List[str] = []

class Certification(BaseModel):
    name: Optional[str] = None
    issuer: Optional[str] = None
    year: Optional[str] = None

class ParsedResume(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    linkedIn: Optional[str] = None
    github: Optional[str] = None
    portfolio: Optional[str] = None
    summary: Optional[str] = None
    skills: List[str] = []
    experience: List[Experience] = []
    education: List[Education] = []
    projects: List[Project] = []
    certifications: List[Certification] = []
    aiScore: float = 0.0
    aiSuggestions: List[str] = []

In [29]:
parser = PydanticOutputParser(pydantic_object=ParsedResume)

In [30]:
# initialize model (example — adapt to your stack)
model = init_chat_model(model_provider='openai', model='gpt-4', api_key=OPENAI_API_KEY)

In [31]:
def pdf_loader(state: StateSchema) -> StateSchema:
    url = state['resume_url']
    loader = PyPDFLoader(url)
    documents = loader.load()
    full_text = "\n".join([doc.page_content for doc in documents])
    state['resume_text'] = full_text
    return state

In [32]:
data = pdf_loader({ 'resume_url': 'https://res.cloudinary.com/hiregenx/image/upload/v1758444270/s3orzdkufvum3tnydcva.pdf' })

In [33]:
data

{'resume_url': 'https://res.cloudinary.com/hiregenx/image/upload/v1758444270/s3orzdkufvum3tnydcva.pdf',
 'resume_text': 'MUHAMMAD AHMAD\nFULL STACK DEVELOPER\nLahore, Pakistan ahmaddev477@gmail.com + 92 3412392664 LinkedIn Github\nSUMMARY\nComputer Science undergraduate (6th semester) at COMSATS University Islamabad, Lahore with hands-on experience in\nFullStack development using JavaScript, React.js, Node.js, FastApi and MongoDB. Passionate about building scalable web\napplications and exploring applications of AI/ML (GEN AI). Actively seeking an internship opportunity to apply my skills and\ngrow in a\nprofessional environment.\nSKILLS\nProgramming languages: JavaScript, Python, Rust\nDatabase: MySQL, PostgreSQL, MongoDB.\nFrontend Technologies: React Js, Tailwind Css.\nBackend Technologies: Node.Js, Express Js, FastApi.\nVersion Control: Git.\nSoft Skills: Problem-Solving,Leadership Qualities, Time Management, Teamwork.\nPROJECT EXPERIENCE\nFull Stack Developer (Freelance)\nRemote |

In [34]:
def resume_parser(state: StateSchema) -> StateSchema:
    print("Parsing Resume...")
    prompt = prompt_template.format(
        resume_text=state['resume_text'],
        format_instructions=parser.get_format_instructions()
    )
    result = model.invoke(prompt)
    # print("RESULT ", result.content)
    parsed_output = parser.parse(result.content)
    state['parsed_resume'] = parsed_output.model_dump()
    print("Result", state['parsed_resume'])
    return state

In [35]:
result = resume_parser(data)

Parsing Resume...
Result {'name': 'Muhammad Ahmad', 'email': 'ahmaddev477@gmail.com', 'phone': '+ 92 3412392664', 'linkedIn': None, 'github': None, 'portfolio': None, 'summary': 'Computer Science undergraduate (6th semester) at COMSATS University Islamabad, Lahore with hands-on experience in FullStack development using JavaScript, React.js, Node.js, FastApi and MongoDB. Passionate about building scalable web applications and exploring applications of AI/ML (GEN AI). Actively seeking an internship opportunity to apply my skills and grow in a professional environment.', 'skills': ['JavaScript', 'Python', 'Rust', 'MySQL', 'PostgreSQL', 'MongoDB', 'React Js', 'Tailwind Css', 'Node.Js', 'Express Js', 'FastApi', 'Git', 'Problem-Solving', 'Leadership Qualities', 'Time Management', 'Teamwork'], 'experience': [{'company': 'Freelance', 'position': 'Full Stack Developer', 'startDate': 'Jan 2024', 'endDate': 'Present', 'description': 'Developed a number of web applications including onyxrenders.co